<img src="http://imgur.com/1ZcRyrc.png" style="float: left; margin: 20px; height: 55px">

# Recommender Systems

_Authors:_ Riley Dallas (mostly), Tim Book

---

In [1]:
import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.metrics.pairwise import pairwise_distances

## Load `movies.csv` and `ratings.csv`
---

We'll be using the [MovieLens](https://grouplens.org/datasets/movielens/) dataset for building our recommendation engine. There are two CSVs (`movies.csv` and `ratings.csv`) that we'll eventually inner join. 

In [2]:
movies = pd.read_csv('datasets/movies.csv')
movies.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [3]:
ratings = pd.read_csv('datasets/ratings.csv')
ratings.head()

,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


# Merge 'em!

In [4]:
# YOU: Merge these datasets. What kind of merge will it be?
df = pd.merge(ratings, movies, on='movieId', how='left')
df.head()

,userId,movieId,rating,timestamp,title,genres
0,1,1,4.0,964982703,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,1,3,4.0,964981247,Grumpier Old Men (1995),Comedy|Romance
2,1,6,4.0,964982224,Heat (1995),Action|Crime|Thriller
3,1,47,5.0,964983815,Seven (a.k.a. Se7en) (1995),Mystery|Thriller
4,1,50,5.0,964982931,"Usual Suspects, The (1995)",Crime|Mystery|Thriller


In [5]:
# Always after a merge, make sure the row counts make sense!
print(ratings.shape)
print(movies.shape)
print(df.shape)

(100836, 4)
(9742, 3)
(100836, 6)


# Drop 'em!
We really only need the user IDs, ratings, and movie title. Drop everything else.

In [6]:
df.drop(columns=['timestamp', 'movieId', 'genres'], inplace=True)

In [7]:
df.head()

,userId,rating,title
0,1,4.0,Toy Story (1995)
1,1,4.0,Grumpier Old Men (1995)
2,1,4.0,Heat (1995)
3,1,5.0,Seven (a.k.a. Se7en) (1995)
4,1,5.0,"Usual Suspects, The (1995)"


## Create pivot table
---

Because we're creating an item-based collaborative recommender (where item in this case is our movies), we'll set up our pivot table as follows:
1. The `title` will be the index
2. The `userId` will be the column
3. The `rating` will be the value

**If we were building a user-based collaborative recommender, what would change about this pivot table?**

In [8]:
rating_mx = pd.pivot_table(
    df,
    index='title',
    columns='userId',
    values='rating'
)

rating_mx.head()

# Optionally mean "center"
#  - Note that broadcasting an ndarray onto a 2D matrix does not work
#     automatically with a Series, hence must reshape to a col vector
# rating_mx - 3

userId,1,2,3,4,5,6,7,8,9,10,...,601,602,603,604,605,606,607,608,609,610
title,,,,,,,,,,,,,,,,,,,,,
'71 (2014),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.0
'Hellboy': The Seeds of Creation (2004),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
'Round Midnight (1986),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
'Salem's Lot (2004),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
'Til There Was You (1997),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [9]:
rating_mx.shape

(9719, 610)

## Create sparse matrix
---

In a minute, we'll calculate the cosine similarity for each movie using the `pairwise_distances` function. Before that, we need to create a sparse matrix (datatype) using `scipy`'s `sparse` module like so:
```python
sparse.csr_matrix(pivot.fillna(0), metric='cosine')
```

In [10]:
ratings_sparse = sparse.csr_matrix(rating_mx.fillna(0))
# print(ratings_sparse)

## Calculate cosine similarity
---

`sklearn` has a built-in `pairwise_distances` function that we can use for our recommender. It will return a square matrix, comparing every movie with every other movie in the dataset.

```python
pairwise_distances(sparse_pivot, metric='cosine')
cosine_distances(sparse_pivot)                     # Identical but more concise
```

In [11]:
# Note that a distance of 1 is a similarity of 0.
dists = pairwise_distances(ratings_sparse, metric='cosine')
# Identical but more concise
# dists = cosine_distances(ratings_sparse)

np.round(dists[:10, :10], 2)

array([[0.  , 1.  , 1.  , 1.  , 1.  , 1.  , 1.  , 1.  , 0.86, 1.  ],
       [1.  , 0.  , 0.29, 1.  , 1.  , 1.  , 1.  , 1.  , 1.  , 1.  ],
       [1.  , 0.29, 0.  , 1.  , 1.  , 1.  , 0.82, 1.  , 1.  , 1.  ],
       [1.  , 1.  , 1.  , 0.  , 0.14, 1.  , 1.  , 1.  , 1.  , 1.  ],
       [1.  , 1.  , 1.  , 0.14, 0.  , 1.  , 1.  , 1.  , 1.  , 1.  ],
       [1.  , 1.  , 1.  , 1.  , 1.  , 0.  , 1.  , 1.  , 1.  , 1.  ],
       [1.  , 1.  , 0.82, 1.  , 1.  , 1.  , 0.  , 0.93, 0.9 , 0.75],
       [1.  , 1.  , 1.  , 1.  , 1.  , 1.  , 0.93, 0.  , 1.  , 1.  ],
       [0.86, 1.  , 1.  , 1.  , 1.  , 1.  , 0.9 , 1.  , 0.  , 0.84],
       [1.  , 1.  , 1.  , 1.  , 1.  , 1.  , 0.75, 1.  , 0.84, 0.  ]])

In [12]:
dists.shape

(9719, 9719)

## Create distances DataFrame
---
At this point, we essentially have a recommender! We'll load it into a `pandas` DataFrame for ease-of-use. 

In [13]:
df_rec = pd.DataFrame(dists, columns=rating_mx.index, index=rating_mx.index)
df_rec.iloc[:5, :5]

title,'71 (2014),'Hellboy': The Seeds of Creation (2004),'Round Midnight (1986),'Salem's Lot (2004),'Til There Was You (1997)
title,,,,,
'71 (2014),0.0,1.000000,1.000000,1.000000,1.000000
'Hellboy': The Seeds of Creation (2004),1.0,0.000000,0.292893,1.000000,1.000000
'Round Midnight (1986),1.0,0.292893,0.000000,1.000000,1.000000
'Salem's Lot (2004),1.0,1.000000,1.000000,0.000000,0.142507
'Til There Was You (1997),1.0,1.000000,1.000000,0.142507,0.000000


# Time to Play!
We've got our recommender - let's try out a few movies.

In [14]:
titles = rating_mx.index
titles[titles.str.lower().str.contains('lord of the rings')]

Index(['Lord of the Rings, The (1978)',
       'Lord of the Rings: The Fellowship of the Ring, The (2001)',
       'Lord of the Rings: The Return of the King, The (2003)',
       'Lord of the Rings: The Two Towers, The (2002)'],
      dtype='object', name='title')

In [15]:
df_rec['Lord of the Rings: The Fellowship of the Ring, The (2001)'].sort_values().head()

title
Lord of the Rings: The Fellowship of the Ring, The (2001)        0.000000
Lord of the Rings: The Return of the King, The (2003)            0.125600
Lord of the Rings: The Two Towers, The (2002)                    0.125868
Pirates of the Caribbean: The Curse of the Black Pearl (2003)    0.338440
Matrix, The (1999)                                               0.345872
Name: Lord of the Rings: The Fellowship of the Ring, The (2001), dtype: float64

In [16]:
df_rec['John Wick (2014)'].sort_values().head()

title
John Wick (2014)                                  0.000000
Mad Max: Fury Road (2015)                         0.361477
Snowpiercer (2013)                                0.381227
Fast Five (Fast and the Furious 5, The) (2011)    0.434396
Rogue One: A Star Wars Story (2016)               0.434650
Name: John Wick (2014), dtype: float64

In [17]:
df_rec['Shrek (2001)'].sort_values().head()

title
Shrek (2001)               0.000000
Finding Nemo (2003)        0.298565
Monsters, Inc. (2001)      0.318305
Incredibles, The (2004)    0.320871
Shrek 2 (2004)             0.325528
Name: Shrek (2001), dtype: float64

In [18]:
# What the hell?
df_rec['Frozen (2010)'].sort_values().head()

title
Frozen (2010)                                  0.000000
Paranormal Activity 3 (2011)                   0.002946
Dark Skies (2013)                              0.078365
'71 (2014)                                     0.240743
One-Armed Swordsman, The (Dubei dao) (1967)    0.240743
Name: Frozen (2010), dtype: float64

# Conclusions
Recommender systems are cool, and there are lots of ways to create them! Which technique you pick is often determined by what kind of rating data (if any) you have.

### Types of Recommenders
1. Collaborative filtering (what we did today!)
1. Content-based recommenders (use information about the items themselves)
1. Knowledge-based recommenders (use user-input to make informed decisions)
1. Hybrid or ensemble recommenders (some combination of 1-3 above)